# Notebook 04 — Amazonas Two-City Case Study

## Humaitá vs São Gabriel da Cachoeira

This notebook creates a focused structured case study for two municipalities in Amazonas state:

1. **Humaitá**, located in the southern/southeastern portion of Amazonas.
2. **São Gabriel da Cachoeira**, located in the northwestern portion of Amazonas.

The workflow is:

```text
two-city MapBiomas-style table
+ two-city INPE / PRODES-style table
→ municipality-year environmental panel
→ native vegetation loss
→ agriculture/pasture expansion
→ environmental screening score
→ comparison and interpretation
```

Important:

> The data in this notebook is synthetic and illustrative. It validates the workflow, but it should be replaced by real MapBiomas and INPE / PRODES values before any real environmental analysis.


## 1. Setup

Run this notebook from either the project root or from the `notebooks/` folder.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Project root:", PROJECT_ROOT)
print("Source directory:", SRC_DIR)


## 2. Load project paths

This notebook writes focused case-study outputs to:

```text
data/processed/
reports/figures/
```


In [ ]:
from agro_rag.utils.paths import (
    ensure_project_directories,
    CLEANED_MAPBIOMAS_DIR,
    CLEANED_INPE_DIR,
    PROCESSED_DATA_DIR,
    FIGURES_DIR,
)

ensure_project_directories()

print("Cleaned MapBiomas directory:", CLEANED_MAPBIOMAS_DIR)
print("Cleaned INPE directory:", CLEANED_INPE_DIR)
print("Processed data directory:", PROCESSED_DATA_DIR)
print("Figures directory:", FIGURES_DIR)


## 3. Define the two municipalities

We use IBGE municipality codes:

| Municipality | State | Code |
|---|---:|---:|
| Humaitá | AM | `1301704` |
| São Gabriel da Cachoeira | AM | `1303809` |


In [ ]:
MUNICIPALITIES = [
    {
        "municipality_code": "1301704",
        "municipality_name": "Humaitá",
        "state": "AM",
        "biome": "Amazon",
        "region_note": "Southern / southeastern Amazonas",
    },
    {
        "municipality_code": "1303809",
        "municipality_name": "São Gabriel da Cachoeira",
        "state": "AM",
        "biome": "Amazon",
        "region_note": "Northwestern Amazonas",
    },
]

municipalities_df = pd.DataFrame(MUNICIPALITIES)
municipalities_df


## 4. Create synthetic MapBiomas-style data

The synthetic pattern represents a contrast:

- Humaitá has stronger synthetic land-use pressure.
- São Gabriel da Cachoeira has weaker synthetic land-use pressure.

These values are **not real data**.


In [ ]:
def build_two_city_mapbiomas_table() -> pd.DataFrame:
    configs = {
        "1301704": {
            "municipality_name": "Humaitá",
            "state": "AM",
            "biome": "Amazon",
            "native_base": 620_000,
            "agriculture_base": 32_000,
            "pasture_base": 95_000,
            "native_loss_rate": 0.018,
            "agriculture_growth_rate": 0.030,
            "pasture_growth_rate": 0.025,
        },
        "1303809": {
            "municipality_name": "São Gabriel da Cachoeira",
            "state": "AM",
            "biome": "Amazon",
            "native_base": 1_250_000,
            "agriculture_base": 4_500,
            "pasture_base": 11_000,
            "native_loss_rate": 0.003,
            "agriculture_growth_rate": 0.008,
            "pasture_growth_rate": 0.006,
        },
    }

    years = list(range(2018, 2025))
    rows = []

    for municipality_code, cfg in configs.items():
        for i, year in enumerate(years):
            native_area = cfg["native_base"] * (1 - cfg["native_loss_rate"] * i)
            agriculture_area = cfg["agriculture_base"] * (1 + cfg["agriculture_growth_rate"] * i)
            pasture_area = cfg["pasture_base"] * (1 + cfg["pasture_growth_rate"] * i)
            forest_area = native_area * 0.78

            rows.append(
                {
                    "municipality_code": municipality_code,
                    "municipality_name": cfg["municipality_name"],
                    "state": cfg["state"],
                    "biome": cfg["biome"],
                    "year": year,
                    "mapbiomas_forest_area_ha": round(forest_area, 2),
                    "mapbiomas_native_vegetation_area_ha": round(native_area, 2),
                    "mapbiomas_agriculture_area_ha": round(agriculture_area, 2),
                    "mapbiomas_pasture_area_ha": round(pasture_area, 2),
                }
            )

    return pd.DataFrame(rows)


mapbiomas_two_city_df = build_two_city_mapbiomas_table()

mapbiomas_two_city_path = (
    CLEANED_MAPBIOMAS_DIR / "amazonas_two_city_mapbiomas_municipality_year.parquet"
)
mapbiomas_two_city_df.to_parquet(mapbiomas_two_city_path, index=False)

print("Rows:", len(mapbiomas_two_city_df))
print("Saved to:", mapbiomas_two_city_path)

mapbiomas_two_city_df.head(10)


## 5. Create synthetic INPE / PRODES-style data

This creates a stronger deforestation signal in Humaitá and a weaker signal in São Gabriel da Cachoeira.

These are illustrative values only.


In [ ]:
def build_two_city_inpe_table() -> pd.DataFrame:
    base_deforestation = {
        "1301704": 850,
        "1303809": 75,
    }

    annual_trend = {
        "1301704": 85,
        "1303809": 6,
    }

    shock_2022 = {
        "1301704": 180,
        "1303809": 15,
    }

    rows = []

    for municipality_code in base_deforestation:
        for i, year in enumerate(range(2018, 2025)):
            value = base_deforestation[municipality_code] + annual_trend[municipality_code] * i

            if year == 2022:
                value += shock_2022[municipality_code]

            rows.append(
                {
                    "municipality_code": municipality_code,
                    "year": year,
                    "inpe_prodes_deforestation_area_ha": round(value, 2),
                }
            )

    return pd.DataFrame(rows)


inpe_two_city_df = build_two_city_inpe_table()

inpe_two_city_path = CLEANED_INPE_DIR / "amazonas_two_city_inpe_prodes_municipality_year.parquet"
inpe_two_city_df.to_parquet(inpe_two_city_path, index=False)

print("Rows:", len(inpe_two_city_df))
print("Saved to:", inpe_two_city_path)

inpe_two_city_df.head(10)


## 6. Build the two-city environmental panel

This uses the same production module used by Notebook 02.

Output:

```text
data/processed/amazonas_two_city_panel.parquet
```


In [ ]:
from agro_rag.structured.build_environmental_panel import build_environmental_panel_file

amazonas_panel_path = PROCESSED_DATA_DIR / "amazonas_two_city_panel.parquet"

amazonas_panel_df = build_environmental_panel_file(
    mapbiomas_path=mapbiomas_two_city_path,
    inpe_path=inpe_two_city_path,
    output_path=amazonas_panel_path,
    how="left",
)

print("Rows:", len(amazonas_panel_df))
print("Saved to:", amazonas_panel_path)

amazonas_panel_df.head(10)


## 7. Validate the panel

We check municipality count, year range, duplicated keys and missing values.


In [ ]:
print("Shape:", amazonas_panel_df.shape)
print("Municipalities:", amazonas_panel_df["municipality_name"].nunique())
print("Years:", amazonas_panel_df["year"].min(), "to", amazonas_panel_df["year"].max())

duplicated_keys = amazonas_panel_df.duplicated(
    subset=["municipality_code", "year"]
).sum()

print("Duplicated municipality-year records:", duplicated_keys)

missing_summary = amazonas_panel_df.isna().sum()
missing_summary[missing_summary > 0]


## 8. Calculate environmental screening score

Output:

```text
data/processed/amazonas_two_city_panel_scored.parquet
```

Remember:

> This score is a screening indicator only. It is not a legal conclusion.


In [ ]:
from agro_rag.structured.risk_score import score_environmental_panel_file

amazonas_scored_path = PROCESSED_DATA_DIR / "amazonas_two_city_panel_scored.parquet"

amazonas_scored_df = score_environmental_panel_file(
    input_path=amazonas_panel_path,
    output_path=amazonas_scored_path,
    group_columns=["municipality_code"],
    year_column="year",
)

print("Rows:", len(amazonas_scored_df))
print("Saved to:", amazonas_scored_path)

amazonas_scored_df.head()


## 9. Inspect the scored panel

Key fields are the source indicators, derived indicators and screening score.


In [ ]:
score_columns = [
    "municipality_code",
    "municipality_name",
    "state",
    "biome",
    "year",
    "inpe_prodes_deforestation_area_ha",
    "mapbiomas_native_vegetation_area_ha",
    "mapbiomas_native_vegetation_loss_ha",
    "mapbiomas_agriculture_area_ha",
    "mapbiomas_pasture_area_ha",
    "mapbiomas_agriculture_or_pasture_expansion_ha",
    "environmental_risk_score",
    "environmental_risk_class",
]

amazonas_scored_df[score_columns]


## 10. Compare latest year

This cell compares both municipalities in the latest year available.


In [ ]:
latest_year = amazonas_scored_df["year"].max()

latest_comparison = (
    amazonas_scored_df
    .loc[amazonas_scored_df["year"] == latest_year, score_columns]
    .sort_values("environmental_risk_score", ascending=False)
    .reset_index(drop=True)
)

print("Latest year:", latest_year)
latest_comparison


## 11. Aggregate comparison across the full period

This summarizes the multi-year synthetic pattern by municipality.


In [ ]:
comparison_summary = (
    amazonas_scored_df
    .groupby(["municipality_code", "municipality_name", "state", "biome"], as_index=False)
    .agg(
        mean_risk_score=("environmental_risk_score", "mean"),
        max_risk_score=("environmental_risk_score", "max"),
        latest_risk_score=("environmental_risk_score", "last"),
        total_prodes_deforestation_ha=("inpe_prodes_deforestation_area_ha", "sum"),
        total_native_vegetation_loss_ha=("mapbiomas_native_vegetation_loss_ha", "sum"),
        total_agri_pasture_expansion_ha=("mapbiomas_agriculture_or_pasture_expansion_ha", "sum"),
    )
    .sort_values("latest_risk_score", ascending=False)
)

comparison_summary


## 12. Plot environmental screening score over time

The figure is saved to:

```text
reports/figures/amazonas_two_city_risk_score.png
```


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))

for municipality_name, group in amazonas_scored_df.groupby("municipality_name"):
    group = group.sort_values("year")
    ax.plot(
        group["year"],
        group["environmental_risk_score"],
        marker="o",
        linewidth=2,
        label=municipality_name,
    )

ax.set_title("Environmental Screening Score — Humaitá vs São Gabriel da Cachoeira")
ax.set_xlabel("Year")
ax.set_ylabel("Environmental risk score")
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
ax.legend(title="Municipality")

figure_path = FIGURES_DIR / "amazonas_two_city_risk_score.png"
fig.savefig(figure_path, dpi=200, bbox_inches="tight")

plt.show()

print("Figure saved to:", figure_path)


## 13. Plot component indicators

This chart compares the latest-year values for the three score components.


In [ ]:
component_columns = [
    "inpe_prodes_deforestation_area_ha",
    "mapbiomas_native_vegetation_loss_ha",
    "mapbiomas_agriculture_or_pasture_expansion_ha",
]

latest_components = latest_comparison[
    ["municipality_name"] + component_columns
].set_index("municipality_name")

ax = latest_components.plot(kind="bar", figsize=(10, 6))

ax.set_title(f"Latest-Year Screening Components — {latest_year}")
ax.set_xlabel("Municipality")
ax.set_ylabel("Hectares")
ax.grid(True, axis="y", alpha=0.3)
ax.legend(title="Indicator", bbox_to_anchor=(1.05, 1), loc="upper left")

component_figure_path = FIGURES_DIR / "amazonas_two_city_components_latest_year.png"
plt.savefig(component_figure_path, dpi=200, bbox_inches="tight")
plt.show()

print("Figure saved to:", component_figure_path)


## 14. Build interpretation text

This text is designed to be copied into a report or used as structured evidence for a RAG risk summary.


In [ ]:
def classify_comparison(row: pd.Series) -> str:
    score = row["latest_risk_score"]

    if score >= 0.67:
        return "higher screening signal"
    if score >= 0.34:
        return "intermediate screening signal"
    return "lower screening signal"


comparison_summary["screening_interpretation"] = comparison_summary.apply(
    classify_comparison,
    axis=1,
)

comparison_summary


In [ ]:
def build_city_comparison_text(summary_df: pd.DataFrame, latest_year: int) -> str:
    lines = []

    lines.append(f"Amazonas two-city environmental screening comparison — latest year: {latest_year}")
    lines.append("")
    lines.append("Municipalities included:")
    lines.append("- Humaitá, AM, municipality code 1301704")
    lines.append("- São Gabriel da Cachoeira, AM, municipality code 1303809")
    lines.append("")

    for _, row in summary_df.iterrows():
        lines.append(f"{row['municipality_name']} ({row['state']})")
        lines.append(f"- Mean risk score: {row['mean_risk_score']:.3f}")
        lines.append(f"- Maximum risk score: {row['max_risk_score']:.3f}")
        lines.append(f"- Latest risk score: {row['latest_risk_score']:.3f}")
        lines.append(f"- Total synthetic INPE / PRODES deforestation area (ha): {row['total_prodes_deforestation_ha']:.2f}")
        lines.append(f"- Total synthetic native vegetation loss (ha): {row['total_native_vegetation_loss_ha']:.2f}")
        lines.append(f"- Total synthetic agriculture/pasture expansion (ha): {row['total_agri_pasture_expansion_ha']:.2f}")
        lines.append(f"- Interpretation: {row['screening_interpretation']}")
        lines.append("")

    lines.append("Important limitation:")
    lines.append(
        "The data in this notebook is synthetic and illustrative. The results validate the workflow, "
        "but they must not be interpreted as real evidence about these municipalities."
    )
    lines.append(
        "The environmental screening score is not proof of illegal deforestation, legal non-compliance, "
        "environmental crime or final ESG classification."
    )

    return "\n".join(lines)


comparison_text = build_city_comparison_text(
    summary_df=comparison_summary,
    latest_year=latest_year,
)

print(comparison_text)


## 15. Save comparison outputs

This cell saves a compact comparison CSV and the interpretation text.


In [ ]:
comparison_summary_path = PROCESSED_DATA_DIR / "amazonas_two_city_comparison_summary.csv"
comparison_text_path = PROCESSED_DATA_DIR / "amazonas_two_city_interpretation.txt"

comparison_summary.to_csv(comparison_summary_path, index=False)
comparison_text_path.write_text(comparison_text, encoding="utf-8")

print("Comparison summary saved to:", comparison_summary_path)
print("Interpretation text saved to:", comparison_text_path)


## 16. Optional: build a risk summary prompt

This cell builds the messages that could be sent to an LLM.

It does not call the model. It only verifies that the structured evidence can be injected into the risk summary prompt.


In [ ]:
from agro_rag.rag.prompts import build_risk_summary_messages

question = (
    "Compare Humaitá and São Gabriel da Cachoeira using the available structured "
    "environmental screening indicators."
)

demo_context = (
    "Document context would be retrieved from the vector store. "
    "This notebook focuses on structured municipality-year evidence."
)

messages = build_risk_summary_messages(
    question=question,
    structured_evidence=comparison_text,
    context=demo_context,
)

for message in messages:
    print("=" * 80)
    print("ROLE:", message["role"])
    print(message["content"][:1800])
    print()


## 17. Files generated

This notebook generated:

```text
data/interim/cleaned_mapbiomas/amazonas_two_city_mapbiomas_municipality_year.parquet
data/interim/cleaned_inpe/amazonas_two_city_inpe_prodes_municipality_year.parquet
data/processed/amazonas_two_city_panel.parquet
data/processed/amazonas_two_city_panel_scored.parquet
data/processed/amazonas_two_city_comparison_summary.csv
data/processed/amazonas_two_city_interpretation.txt
reports/figures/amazonas_two_city_risk_score.png
reports/figures/amazonas_two_city_components_latest_year.png
```


In [ ]:
generated_files = [
    mapbiomas_two_city_path,
    inpe_two_city_path,
    amazonas_panel_path,
    amazonas_scored_path,
    comparison_summary_path,
    comparison_text_path,
    figure_path,
    component_figure_path,
]

print("Generated files:")
for path in generated_files:
    print("-", path, "| exists:", path.exists())


## 18. Final interpretation

This notebook demonstrates how the project can compare two municipalities through a structured environmental screening workflow.

For this synthetic demonstration, the pattern is intentionally designed so that:

- Humaitá shows a stronger environmental screening signal.
- São Gabriel da Cachoeira shows a weaker environmental screening signal.

Because the data is synthetic, this is only a technical validation of the pipeline.

For real analysis, replace the synthetic values with real MapBiomas and INPE / PRODES municipality-year data and rerun the notebook.

The final principle remains:

> The system supports environmental screening and evidence organization. It does not prove illegal deforestation, assign responsibility or replace expert geospatial, environmental or legal analysis.
